## 整理释读任务的数据集

In [ ]:
import json
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
import cv2
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.data_utils.utils import get_tight_box

sam2_checkpoint = "sam2_logs/configs/sam2.1_training_highres/sam2.1_hiera_l_OBIMD_facs/checkpoints/checkpoint_5.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l_highres.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda')
predictor = SAM2ImagePredictor(sam2_model)

with open('data/OBIMD_cls/class.txt', 'r') as f:
    topclass = set([l.strip() for l in f.readlines()])
with open("data/OBIMD_raw_hj/train.txt") as f:
    train_list = set([l.strip() for l in f.readlines()])
with open("data/OBIMD_raw_hj/label_filt_train.json") as f:
    datas = json.load(f)
    
for image_id, data in tqdm(enumerate(datas)):
    image_path = os.path.basename(data['Rubbing'])
    if image_path.split('.')[0] not in train_list:
        continue
    
    input_box = []
    labels = []
    for sentence in data['RecordUtilSentenceGroupVoList']:
        for char in sentence["RecordUtilOracleCharVoList"]:
            if char['Label'] in topclass:
                if os.path.exists(f'data/OBIMD_cls/test_hj/{char["Label"]}/{image_path.split(".")[0]}_{len(input_box)}.png'):
                    continue
                x, y, w, h = list(map(int, char['Position'].split(',')))
                input_box.append([x, y, x + w, y + h])
                labels.append(char['Label'])
    input_box = np.array(input_box).reshape(-1, 4)
    if len(input_box) == 0:
        continue

    facs = cv2.imread(os.path.join('data/OBIMD_raw_hj/facsimile', image_path))
    predictor.set_image(facs)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )
    if len(masks) == 1:
        masks = masks[None]
    masks = masks.astype(np.uint8) * 255

    for idx, (mask, label) in enumerate(zip(masks, labels)):
        if not mask.any():
            continue
        os.makedirs(f'data/OBIMD_cls/train/{label}', exist_ok=True)
        x, y, w, h = get_tight_box(mask[0])
        cv2.imwrite(f'data/OBIMD_cls/train/{label}/{image_path.split(".")[0]}_{idx}.png', mask[0][y:y+h, x:x+w])

In [ ]:
import json
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
import cv2
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.data_utils.utils import get_tight_box

sam2_checkpoint = "sam2_logs/configs/sam2.1_training_highres/sam2.1_hiera_l_OBIMD_facs/checkpoints/checkpoint_5.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l_highres.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda')
predictor = SAM2ImagePredictor(sam2_model)
with open('data/OBIMD_cls/class.txt', 'r') as f:
    topclass = set([l.strip() for l in f.readlines()])
with open("data/OBIMD_raw_hj/test.txt") as f:
    test_list = set([l.strip() for l in f.readlines()])
with open("data/OBIMD_raw_hj/label_filt_train.json") as f:
    datas = json.load(f)

for image_id, data in tqdm(enumerate(datas)):
    image_path = os.path.basename(data['Rubbing'])
    if image_path.split('.')[0] not in test_list:
        continue
    
    input_box = []
    labels = []
    for sentence in data['RecordUtilSentenceGroupVoList']:
        for char in sentence["RecordUtilOracleCharVoList"]:
            if char['Label'] in topclass:
                if os.path.exists(f'data/OBIMD_cls/test_hj/{char["Label"]}/{image_path.split(".")[0]}_{len(input_box)}.png'):
                    continue
                x, y, w, h = list(map(int, char['Position'].split(',')))
                input_box.append([x, y, x + w, y + h])
                labels.append(char['Label'])
    input_box = np.array(input_box).reshape(-1, 4)
    if len(input_box) == 0:
        continue

    facs = cv2.imread(os.path.join('data/OBIMD_raw_hj/facsimile', image_path))
    predictor.set_image(facs)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )
    if len(masks) == 1:
        masks = masks[None]
    masks = masks.astype(np.uint8) * 255

    for idx, (mask, label) in enumerate(zip(masks, labels)):
        if not mask.any():
            continue
        os.makedirs(f'data/OBIMD_cls/test_hj/{label}', exist_ok=True)
        x, y, w, h = get_tight_box(mask[0])
        cv2.imwrite(f'data/OBIMD_cls/test_hj/{label}/{image_path.split(".")[0]}_{idx}.png', mask[0][y:y+h, x:x+w])

In [ ]:
import json
import os
from tqdm import tqdm
import cv2
import numpy as np
from pycocotools import mask as maskUtils
from sam2.data_utils.utils import get_tight_box

os.makedirs('data/OBIMD_cls_test100/test_hj', exist_ok=True)
for file in tqdm(os.listdir('data/OBIMD_test100/facsimile_json')):
    base_name = file.split('.')[0]
    with open(os.path.join('data/OBIMD_test100/facsimile_json', file)) as f:
        data = json.load(f)
    for idx, ann in enumerate(data['annotations']):
        x, y, w, h = ann['bbox']
        mask = maskUtils.decode(ann['segmentation'])
        mask = mask[y:y+h, x:x+w]
        cv2.imwrite(f'data/OBIMD_cls_test100/test_hj/{base_name}_{idx}.png', mask * 255)

In [1]:
import json
import os
from tqdm import tqdm
import cv2
import numpy as np
from pycocotools import mask as maskUtils
from sam2.data_utils.utils import get_tight_box
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "sam2_logs/configs/sam2.1_baseline/sam2.1_hiera_l_OBIMD_charformer.yaml/checkpoints/checkpoint_10.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l_highres.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda:0')
predictor = SAM2ImagePredictor(sam2_model)
exp_name = "charformer"
os.makedirs(f'data/OBIMD_cls_test100/{exp_name}', exist_ok=True)

for file in tqdm(os.listdir('data/OBIMD_test100/facsimile_json')):
    base_name = file.split('.')[0]
    with open(os.path.join('data/OBIMD_test100/facsimile_json', file)) as f:
        data = json.load(f)
    
    input_box = []
    for idx, ann in enumerate(data['annotations']):
        x, y, w, h = ann['bbox']
        mask = maskUtils.decode(ann['segmentation'])
        mask = mask[y:y+h, x:x+w]
        input_box.append([x, y, x + w, y + h])
    input_box = np.array(input_box).reshape(-1, 4)
    if len(input_box) == 0:
        continue
    
    image = cv2.imread(os.path.join('data/OBIMD_test100/rubbing', base_name + '.jpg'))
    predictor.set_image(image)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box, # use gt box here
        multimask_output=False,
    )
    if len(masks) == 1:
        masks = masks[None]
    masks = masks.astype(np.uint8) * 255

    for idx, mask in enumerate(masks):
        if not mask.any():
            continue
        x, y, w, h = get_tight_box(mask[0])
        cv2.imwrite(f'data/OBIMD_cls_test100/{exp_name}/{base_name}_{idx}.png', mask[0][y:y+h, x:x+w])

100%|██████████| 100/100 [00:19<00:00,  5.12it/s]


## PSNR + SSIM

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

def calculate_metrics(gt_path: str, pred_path: str, 
                      data_range: int = 255, 
                      image_size: tuple[int, int] = (224, 224)) -> tuple[float, float]:
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.resize(gt, image_size, interpolation=cv2.INTER_LINEAR)
    pred = cv2.resize(pred, image_size, interpolation=cv2.INTER_LINEAR)
    
    gt = gt.astype(np.float32)
    pred = pred.astype(np.float32)
    
    psnr_score = psnr(gt, pred, data_range=data_range)
    ssim_score = ssim(
        gt, pred, 
        data_range=data_range,
        gaussian_weights=True,  # 启用高斯加权
        sigma=1.5,              # 高斯核标准差
        use_sample_covariance=False,  # 使用无偏估计
        multichannel=False       # 单通道灰度图
    )
    return psnr_score, ssim_score

psnr_result, ssim_result = [], []
exp_name = "raw"
for root, dirs, files in tqdm(os.walk(f"data/OBIMD_cls/{exp_name}")):
    for file in files:
        pred_path = os.path.join(root, file)
        gt_path = os.path.join(root.replace(exp_name, "test_hj"), file)
        psnr_score, ssim_score = calculate_metrics(gt_path, pred_path)
        psnr_result.append(psnr_score)
        ssim_result.append(ssim_score)
sum(psnr_result) / len(psnr_result), sum(ssim_result) / len(ssim_result)

In [2]:
# 只在test100上做测试
import cv2
import numpy as np
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

def calculate_metrics(gt_path: str, pred_path: str, 
                      data_range: int = 255, 
                      image_size: tuple[int, int] = (224, 224)) -> tuple[float, float]:
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.resize(gt, image_size, interpolation=cv2.INTER_LINEAR)
    pred = cv2.resize(pred, image_size, interpolation=cv2.INTER_LINEAR)
    
    gt = gt.astype(np.float32)
    pred = pred.astype(np.float32)
    
    psnr_score = psnr(gt, pred, data_range=data_range)
    if np.isinf(psnr_score):
        return
    ssim_score = ssim(
        gt, pred, 
        data_range=data_range,
        gaussian_weights=True,  # 启用高斯加权
        sigma=1.5,              # 高斯核标准差
        use_sample_covariance=False,  # 使用无偏估计
        multichannel=False       # 单通道灰度图
    )
    return psnr_score, ssim_score

psnr_result, ssim_result = [], []
exp_name = "charformer"
for files in tqdm(os.listdir(f"data/OBIMD_cls_test100/test_hj")):
    pred_path = os.path.join(f"data/OBIMD_cls_test100/{exp_name}", files)
    gt_path = os.path.join("data/OBIMD_cls_test100/test_hj", files)
    try:
        psnr_score, ssim_score = calculate_metrics(gt_path, pred_path)
    except:
        continue
    psnr_result.append(psnr_score)
    ssim_result.append(ssim_score)
sum(psnr_result) / len(psnr_result), sum(ssim_result) / len(ssim_result)

100%|██████████| 769/769 [00:07<00:00, 100.45it/s]


(9.243175112330777, 0.6081848256991386)